In [52]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import pyspark
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
from pyspark.sql import SparkSession

from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, fbeta_score, confusion_matrix, ConfusionMatrixDisplay



import numpy as np

import pickle


In [53]:
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score, classification_report

In [54]:
from utils.prepare_data import import_feature_and_label

In [55]:
from utils.data_split import split_oot

In [56]:
import utils.prepare_data
print(dir(utils.prepare_data))


['SparkSession', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'glob', 'import_feature_and_label', 'os']


# set up spark

In [57]:
spark = SparkSession.builder \
    .appName("ModelInputPrep") \
    .config("spark.driver.memory", "16g")\
    .config("spark.executor.memory", "16g")\
    .config("spark.driver.maxResultSize", "4g")\
    .config("spark.network.timeout", "600s")\
    .config("spark.executor.heartbeatInterval", "60s")\
    .getOrCreate()


# prepare data for model train

**import gold data**

In [58]:
x,y = import_feature_and_label('datamart/gold', spark)


In [59]:
x

,customer_id,snapshot_date,age,annual_income,monthly_inhand_salary,num_bank_accounts,num_credit_card,interest_rate,num_of_loan,delay_from_due_date,...,avg_fe_11,avg_fe_12,avg_fe_13,avg_fe_14,avg_fe_15,avg_fe_16,avg_fe_17,avg_fe_18,avg_fe_19,avg_fe_20
0,CUS_0x10c0,2024-02-01,39,49454.128906,4328.177734,8.0,5.0,23.0,2.0,59,...,156.142857,118.214286,74.000000,94.357143,84.285714,44.142857,29.714286,144.714286,89.285714,89.000000
1,CUS_0x12ef,2024-02-01,43,68898.000000,5588.500000,7.0,4.0,11.0,0.0,24,...,145.285714,84.714286,64.928571,96.071429,91.642857,157.571429,83.285714,85.714286,118.357143,81.428571
2,CUS_0x1414,2024-02-01,46,51093.121094,4080.760010,8.0,9.0,25.0,6.0,42,...,95.071429,103.071429,122.142857,91.571429,113.142857,88.785714,136.071429,107.000000,124.428571,79.142857
3,CUS_0x18f4,2024-02-01,23,35805.250000,3106.770752,4.0,7.0,16.0,0.0,24,...,142.500000,125.642857,101.000000,133.785714,116.357143,120.428571,94.785714,83.785714,98.571429,73.071429
4,CUS_0x1935,2024-02-01,44,17096.250000,1669.687500,10.0,8.0,19.0,7.0,56,...,89.428571,67.571429,69.642857,37.071429,140.928571,106.785714,148.071429,95.000000,127.857143,90.214286
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8969,CUS_0xbfb4,2023-09-01,37,17983.019531,1522.584961,5.0,6.0,9.0,4.0,14,...,113.333333,89.777778,123.000000,142.444444,33.111111,60.444444,112.666667,64.333333,132.555556,29.222222
8970,CUS_0xc242,2023-09-01,18,17018.449219,1188.204224,8.0,6.0,25.0,7.0,16,...,85.111111,81.555556,94.888889,114.000000,70.888889,19.888889,51.555556,109.111111,121.111111,120.555556
8971,CUS_0xc26e,2023-09-01,42,52395.839844,4278.319824,6.0,3.0,6.0,4.0,24,...,61.777778,114.666667,81.888889,82.777778,119.444444,122.666667,97.111111,97.000000,121.777778,77.666667
8972,CUS_0xc2cb,2023-09-01,33,22387.125000,1677.593750,4.0,3.0,12.0,0.0,3,...,89.333333,85.777778,122.666667,154.666667,92.444444,103.888889,129.333333,104.444444,77.777778,137.333333


**set up data split config**

In [60]:
# set up config
model_train_date_str = "2024-09-01"  
train_test_period_months = 12
oot_period_months = 2
train_test_ratio = 0.8

config = {}
config["model_train_date_str"] = model_train_date_str
config["train_test_period_months"] = train_test_period_months
config["oot_period_months"] =  oot_period_months
config["model_train_date"] =  datetime.strptime(model_train_date_str, "%Y-%m-%d").date()
config["oot_end_date"] =  config['model_train_date'] - timedelta(days = 1)
config["oot_start_date"] =  config['model_train_date'] - relativedelta(months = oot_period_months)
config["train_test_end_date"] =  config["oot_start_date"] - timedelta(days = 1)
config["train_test_start_date"] =  config["oot_start_date"] - relativedelta(months = train_test_period_months)
config["train_test_ratio"] = train_test_ratio

**split data**

In [61]:
x_traintest, y_traintest, x_oot, y_oot = split_oot(x, y, config)  # split OOT


In [62]:
x_train, x_test, y_train, y_test = train_test_split(x_traintest, y_traintest, 
                                                    test_size=config['train_test_ratio'], 
                                                    random_state=611, 
                                                    shuffle=True, 
                                                    stratify=y_traintest['label'])

**change data format**

In [63]:
# Transform data into numpy arrays
x_train_arr = x_train.drop(columns=['customer_id', 'snapshot_date']).values
x_test_arr = x_test.drop(columns=['customer_id', 'snapshot_date']).values
x_oot_arr = x_oot.drop(columns=['customer_id', 'snapshot_date']).values

y_train_arr = y_train['label'].values
y_test_arr = y_test['label'].values
y_oot_arr = y_oot['label'].values

In [64]:
# Normalize x
scaler = StandardScaler()
x_train_nor = scaler.fit_transform(x_train_arr)
x_test_nor = scaler.transform(x_test_arr)
x_oot_nor = scaler.transform(x_oot_arr)

In [74]:
save_dir = "datamart/model_input"
os.makedirs(save_dir, exist_ok=True)


np.save(os.path.join(save_dir, "x_train_nor.npy"), x_train_nor)
np.save(os.path.join(save_dir, "x_test_nor.npy"), x_test_nor)
np.save(os.path.join(save_dir, "x_oot_nor.npy"), x_oot_nor)


np.save(os.path.join(save_dir, "y_train_arr.npy"), y_train_arr)
np.save(os.path.join(save_dir, "y_test_arr.npy"), y_test_arr)
np.save(os.path.join(save_dir, "y_oot_arr.npy"), y_oot_arr)

# Model Train

In [75]:
base_dir = "datamart/model_input"


x_train_nor = np.load(os.path.join(base_dir, "x_train_nor.npy"))
x_test_nor = np.load(os.path.join(base_dir, "x_test_nor.npy"))
x_oot_nor = np.load(os.path.join(base_dir, "x_oot_nor.npy"))


y_train_arr = np.load(os.path.join(base_dir, "y_train_arr.npy"))
y_test_arr = np.load(os.path.join(base_dir, "y_test_arr.npy"))
y_oot_arr = np.load(os.path.join(base_dir, "y_oot_arr.npy"))

**LogisticRegression**

In [65]:
# Train model
logistic = LogisticRegression()
logistic.fit(x_train_nor, y_train_arr)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [66]:
# Save model

os.makedirs("models", exist_ok=True)

with open("models/logisticmodel.pkl", "wb") as f:
    pickle.dump(logistic, f)

# Model Inference

In [67]:
with open("models/logisticmodel.pkl", "rb") as f:
    loaded_model = pickle.load(f)

In [68]:
y_train_pred = loaded_model.predict(x_train_nor)
y_train_prob = loaded_model.predict_proba(x_train_nor)[:, 1]

In [69]:
print("Train Accuracy:", accuracy_score(y_train_arr, y_train_pred))
print("Train ROC AUC:", roc_auc_score(y_train_arr, y_train_prob))
print("Train Confusion Matrix:\n", confusion_matrix(y_train_arr, y_train_pred))
print("Train Classification Report:\n", classification_report(y_train_arr, y_train_pred))


Train Accuracy: 0.7162048698572628
Train ROC AUC: 0.6311544903022259
Train Confusion Matrix:
 [[842  12]
 [326  11]]
Train Classification Report:
               precision    recall  f1-score   support

           0       0.72      0.99      0.83       854
           1       0.48      0.03      0.06       337

    accuracy                           0.72      1191
   macro avg       0.60      0.51      0.45      1191
weighted avg       0.65      0.72      0.61      1191



In [70]:
y_pred = loaded_model.predict(x_test_nor)
y_prob = loaded_model.predict_proba(x_test_nor)[:, 1] 

In [71]:
print(" Accuracy:", accuracy_score(y_test_arr, y_pred))
print(" ROC AUC:", roc_auc_score(y_test_arr, y_prob))
print(" Confusion Matrix:\n", confusion_matrix(y_test_arr, y_pred))
print(" Classification Report:\n", classification_report(y_test_arr, y_pred))

 Accuracy: 0.7010698552548773
 ROC AUC: 0.48440406846238965
 Confusion Matrix:
 [[3313  105]
 [1320   29]]
 Classification Report:
               precision    recall  f1-score   support

           0       0.72      0.97      0.82      3418
           1       0.22      0.02      0.04      1349

    accuracy                           0.70      4767
   macro avg       0.47      0.50      0.43      4767
weighted avg       0.57      0.70      0.60      4767



In [76]:
save_dir = "datamart/gold/model_inference"
os.makedirs(save_dir, exist_ok=True)

In [77]:
df_train_pred = pd.DataFrame({
    "y_train_pred": y_train_pred,
    "y_train_prob": y_train_prob
})
df_train_pred.to_csv(os.path.join(save_dir, "train_predictions.csv"), index=False)


df_test_pred = pd.DataFrame({
    "y_test_pred": y_pred,
    "y_test_prob": y_prob
})
df_test_pred.to_csv(os.path.join(save_dir, "test_predictions.csv"), index=False)

# Model Monitor

In [72]:
y_oot_pred = loaded_model.predict(x_oot_nor)
y_oot_prob = loaded_model.predict_proba(x_oot_nor)[:, 1]

In [73]:
print("OOT Accuracy:", accuracy_score(y_oot_arr, y_oot_pred))
print("OOT ROC AUC:", roc_auc_score(y_oot_arr, y_oot_prob))
print("OOT Confusion Matrix:\n", confusion_matrix(y_oot_arr, y_oot_pred))
print("OOT Classification Report:\n", classification_report(y_oot_arr, y_oot_pred))


OOT Accuracy: 0.6989032901296112
OOT ROC AUC: 0.4997538515000579
OOT Confusion Matrix:
 [[698  14]
 [288   3]]
OOT Classification Report:
               precision    recall  f1-score   support

           0       0.71      0.98      0.82       712
           1       0.18      0.01      0.02       291

    accuracy                           0.70      1003
   macro avg       0.44      0.50      0.42      1003
weighted avg       0.55      0.70      0.59      1003



In [78]:
monitor_dir = "datamart/gold/model_monitor"
os.makedirs(monitor_dir, exist_ok=True)

In [79]:
df_oot_pred = pd.DataFrame({
    "y_oot_pred": y_oot_pred,
    "y_oot_prob": y_oot_prob
})
df_oot_pred.to_csv(os.path.join(monitor_dir, "oot_predictions.csv"), index=False)
